In [2]:
import sys

import polars as pl
import torch


'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

df = pl.read_parquet(DIR + 'target_dyn_demand_weekly.parquet')

In [3]:
# 타입 정리
df = df.with_columns(
    pl.col("oper_part_no").cast(pl.Utf8),
    pl.col("demand_dt").cast(pl.Int32, strict=False),
    pl.col("demand_qty").cast(pl.Float64),
).sort(["oper_part_no", "demand_dt"])

In [4]:
import numpy as np

# -----------------------------
# 파트별 랜덤 파라미터(불량률, 보증개월) 생성
# -----------------------------
rng = np.random.default_rng(seed=42)

parts = df.select(pl.col("oper_part_no").unique().sort()).to_series().to_list()
n = len(parts)

In [5]:
# 불량률: U(0.01, 0.05)
defect_rates = rng.uniform(0.01, 0.05, size=n)

In [6]:
# Warranty months: {6, 12, 24, 36} 중 균등 샘플
wty_choices = np.array([6, 12, 24, 36], dtype=int)
wty_months = rng.choice(wty_choices, size=n, replace=True)

In [7]:
# 파라미터 테이블(파트 단위)
param_df = pl.DataFrame({
    "oper_part_no": parts,
    "defect_rate": defect_rates,
    "warranty_months": wty_months,
})

### Sample Data 만들기

In [8]:
from datetime import date, timedelta
from modeling_module.utils.date_util import DateUtil

def yyyyww_to_monday(yyyyww: int) -> date:
    y = int(yyyyww // 100)
    w = int(yyyyww % 100)
    # ISO: Monday=1
    return date.fromisocalendar(y, w, 1)

def date_to_yyyyww(d: date) -> int:
    y, w, _ = d.isocalendar()  # (year, week, weekday)
    return int(y * 100 + w)

def add_weeks_to_yyyyww(yyyyww: int, delta_weeks: int) -> int:
    d0 = yyyyww_to_monday(int(yyyyww))
    d1 = d0 + timedelta(weeks=delta_weeks)
    return date_to_yyyyww(d1)

# 주차 환산(간단히 1개월=4주 가정; 더 정밀은 4.345 사용 가능)
param_df = param_df.with_columns(
    (pl.col("warranty_months") * 4).alias("warranty_weeks")
)

# 메인 DF에 조인
dfx = df.join(param_df, on="oper_part_no", how="left")

# -----------------------------
# 1) 즉시형(naive) 모수 추정
#    demand_qty = defect_rate * sales  → sales ≈ demand_qty / defect_rate
# -----------------------------
dfx = dfx.with_columns(
    (pl.col("demand_qty") / pl.col("defect_rate")).alias("base_sales_est_naive")
)


# -----------------------------
# 2) 보증기간-시프트형(origin) 모수 추정
#    t에서 관측된 demand를 (warranty_weeks)만큼 과거 주차로 귀속
#    → 해당 과거 주차의 판매로 간주
# -----------------------------
# 시프트된 기원 주차(판매 귀속 주차)
dfx = dfx.with_columns(
    pl.struct(["demand_dt", "warranty_weeks"]).map_elements(
        lambda s: add_weeks_to_yyyyww(int(s["demand_dt"]), -int(s["warranty_weeks"])),
        return_dtype = pl.Int64
    ).alias("origin_yyyyww")
)


# 해당 귀속 주차의 판매량 기여분 = demand_qty / defect_rate
dfx = dfx.with_columns(
    (pl.col("demand_qty") / pl.col("defect_rate")).alias("origin_sales_contrib")
)

# 파트 × origin_yyyyww로 모아 origin 기준 판매량 집계
origin_sales = (
    dfx
    .group_by(["oper_part_no", "origin_yyyyww"])
    .agg(pl.sum("origin_sales_contrib").alias("base_sales_est_origin"))
    .sort(["oper_part_no", "origin_yyyyww"])
)

sales_df = origin_sales.rename({'origin_yyyyww': 'yyyyww', 'base_sales_est_origin': 'sales_qty'})
demand_df = df.rename({'demand_dt': 'yyyyww'})
final = demand_df.join(sales_df, on = ['oper_part_no', 'yyyyww'], how = 'left').fill_null(0.0)


In [13]:
final = (
    final.with_columns(
        pl.col('oper_part_no').cast(pl.Utf8),
        pl.col('yyyyww').cast(pl.Int32, strict = False)
    )
    .select('oper_part_no', 'yyyyww', 'demand_qty', 'sales_qty')
)

obs_weeks = final.select('oper_part_no', 'yyyyww').unique()

obs_weeks = final.with_columns(
    pl.col('yyyyww').map_elements(lambda x: yyyyww_to_monday(int(x))).alias('week_date')
)

bounds = (
    obs_weeks
    .group_by("oper_part_no")
    .agg(
        pl.min("week_date").alias("start_date"),
        pl.max("week_date").alias("end_date"),
        pl.count().alias("n_obs_weeks"),
    )
    # 전체 주차수: start~end를 1주 간격 그리드로 만들고 길이 계산 (closed="both" 포함)
    .with_columns(
        pl.date_ranges(pl.col("start_date"), pl.col("end_date"),
                       interval="1w", closed="both")
          .arr.len()
          .alias("n_total_weeks")
    )
    .with_columns(
        (pl.col("n_total_weeks") - pl.col("n_obs_weeks"))
            .clip(lower_bound=0)
            .alias("n_empty"),
        pl.col("start_date")
          .map_elements(date_to_yyyyww, return_dtype=pl.Int32)
          .alias("start_yyyyww"),
        pl.col("end_date")
          .map_elements(date_to_yyyyww, return_dtype=pl.Int32)
          .alias("end_yyyyww"),
        (pl.col("n_obs_weeks") / pl.col("n_total_weeks"))
          .alias("coverage_ratio"),
    )
    .select(
        "oper_part_no",
        "start_yyyyww", "end_yyyyww",
        "n_obs_weeks", "n_total_weeks", "n_empty", "coverage_ratio"
    )
    .sort("oper_part_no")
)

bounds


# # 4) (옵션) 실제 누락된 주차 리스트까지 출력하고 싶다면:
# #    - 부품별 start~end 전체 주차 그리드 생성 후 anti-join
# full_grid = (
#     # 부품별 start/end 도출
#     obs_weeks.groupby("oper_part_no").agg(
#         pl.min("week_date").alias("start_date"),
#         pl.max("week_date").alias("end_date")
#     )
#     # 각 부품마다 1주 간격으로 날짜 리스트 생성
#     .with_columns(
#         pl.date_ranges(pl.col("start_date"), pl.col("end_date"), interval="1w").alias("grid_dates")
#     )
#     .explode("grid_dates")
#     .rename({"grid_dates": "week_date"})
# )
#
# # 누락주차 = 전체 그리드에서 관측된 주차를 뺀 것
# missing = (
#     full_grid.join(obs_weeks, on=["oper_part_no", "week_date"], how="anti")
#              .with_columns(pl.col("week_date").map_elements(date_to_yyyyww).alias("yyyyww"))
#              .groupby("oper_part_no")
#              .agg([
#                  pl.count().alias("n_empty"),
#                  pl.col("yyyyww").sort().alias("missing_yyyyww_list")  # 필요 시 확인용
#              ])
#              .sort("oper_part_no")
# )
#
# missing.write_parquet("per_part_missing_weeks.parquet")
# print(missing.head(10))


/var/folders/py/xgf_87rd5nz9rsbc143wp9qc0000gn/T/ipykernel_7544/639757520.py:11: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  obs_weeks = final.with_columns(
/var/folders/py/xgf_87rd5nz9rsbc143wp9qc0000gn/T/ipykernel_7544/639757520.py:21: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n_obs_weeks"),


SchemaError: invalid series dtype: expected `Array`, got `list[date]` for series with name `start_date`

In [ ]:
import math

df = final.with_columns([
    pl.col('oper_part_no').cast(pl.Utf8),
    pl.col('yyyyww').cast(pl.Int32, strict=False),
    pl.col('demand_qty').cast(pl.Float64),
    pl.col('sales_qty').cast(pl.Float64)
]
).fill_null(0.0)

# 그룹 통계 (부품별)
df_stats = final.with_columns([
    pl.mean("demand_qty").over("oper_part_no").alias("d_mean"),
    pl.std("demand_qty").over("oper_part_no").alias("d_std"),
    pl.min("demand_qty").over("oper_part_no").alias("d_min"),
    pl.max("demand_qty").over("oper_part_no").alias("d_max"),

    pl.mean("sales_qty").over("oper_part_no").alias("s_mean"),
    pl.std("sales_qty").over("oper_part_no").alias("s_std"),
    pl.min("sales_qty").over("oper_part_no").alias("s_min"),
    pl.max("sales_qty").over("oper_part_no").alias("s_max"),
]).filter(pl.col('d_mean') > 10)
# 0으로 나눔 방지용 작은 epsilon
eps = 1e-8

# z-score, min_max 정규화 칼럼 생성
df_norm = (df_stats.with_columns([
    # demand
    ((pl.col("demand_qty") - pl.col("d_mean")) / (pl.col("d_std") + eps)).alias("demand_z"),
    ((pl.col("demand_qty") - pl.col("d_min")) / (pl.col("d_max") - pl.col("d_min") + eps)).alias("demand_mm"),

    # sales
    ((pl.col("sales_qty") - pl.col("s_mean")) / (pl.col("s_std") + eps)).alias("sales_z"),
    ((pl.col("sales_qty") - pl.col("s_min")) / (pl.col("s_max") - pl.col("s_min") + eps)).alias("sales_mm"),
]).with_columns([
    ((pl.col("demand_qty") - pl.col("d_mean")) / (pl.col("d_std") + eps)).alias("demand_z"),
    (1.0 / (1.0 + (-pl.col("demand_z")).exp())).alias("demand_z01_sigmoid"),

    ((pl.col('sales_qty') - pl.col('s_mean')) / (pl.col('s_std') + eps)).alias('sales_z'),
    (1.0 / (1.0 + (-pl.col('sales_z')).exp())).alias('sales_z01_sigmoid')
])
           .select([
    "oper_part_no", "yyyyww",
    "demand_qty", "sales_qty",
    "demand_z", "demand_mm",
    "sales_z", "sales_mm",
    'demand_z01_sigmoid', 'sales_z01_sigmoid'
]).fill_null(0.0))

# 2) 로버스트 Min-Max (분위수 기반 0~1, 이상치 완화)
df_norm = df_norm.with_columns(
    pl.col("demand_qty").quantile(0.01).over("oper_part_no").alias("d_q01"),
    pl.col("demand_qty").quantile(0.99).over("oper_part_no").alias("d_q99"),
    pl.col('sales_qty').quantile(0.01).over('oper_part_no').alias('s_q01'),
    pl.col('sales_qty').quantile(0.99).over('oper_part_no').alias('s_q99')
).with_columns(
    # 1~99% 구간으로 윈저 → 0~1
    ((pl.col("demand_qty").clip(pl.col("d_q01"), pl.col("d_q99")) - pl.col("d_q01"))
     / (pl.col("d_q99") - pl.col("d_q01") + eps)).alias("demand_mm_robust"),
    ((pl.col('sales_qty').clip(pl.col('s_q01'), pl.col('s_q99')) - pl.col('s_q01'))
    / (pl.col('s_q99') - pl.col('s_q01') + eps)).alias('sales_mm_robust')
)

# # 3) z→(0,1) 매핑 두 가지
df_norm = df_norm.with_columns(
    # (a) 정규분포 CDF: Φ(z) = 0.5*(1 + erf(z/√2))  → (0,1)
    (0.5 * (1.0 + (pl.col("demand_z") / math.sqrt(2)).map_elements(math.erf, return_dtype=pl.Float64))).alias(
        "demand_z01_cdf"),
    (0.5 * (1.0 + (pl.col("sales_z") / math.sqrt(2)).map_elements(math.erf, return_dtype=pl.Float64))).alias(
        "sales_z01_cdf"),
    # (b) 시그모이드: 1 / (1 + e^{-z})  → (0,1)
    (1.0 / (1.0 + (-pl.col("demand_z")).exp())).alias("demand_z01_sigmoid"),
    (1.0 / (1.0 + (-pl.col("sales_z")).exp())).alias("sales_z01_sigmoid"),




).select([
    "oper_part_no", "yyyyww", "demand_qty", 'sales_qty',
    "demand_z", "demand_mm", "demand_mm_robust", "demand_z01_cdf", "demand_z01_sigmoid",
    'sales_z', 'sales_mm', 'sales_mm_robust', 'sales_z01_cdf', 'sales_z01_sigmoid'
])
df_norm = df_norm.sort(['oper_part_no', 'yyyyww']).with_columns(
    pl.col('yyyyww').cast(pl.Int64).cast(pl.Utf8).alias('yyyyww'))

df_norm

In [ ]:

import matplotlib.pyplot as plt
from tqdm import tqdm

parts = df_norm.select('oper_part_no').unique().to_series().to_list()

show_parts = parts[1:2]
plt.figure(figsize = (32, 12))

for p in tqdm(show_parts):
    print(p)
    gi = df_norm.filter(pl.col('oper_part_no') == p).sort('yyyyww')
    if gi.is_empty(): continue
    gpd = gi.select(['yyyyww', 'demand_z'])
    x = gpd['yyyyww'].to_numpy()
    y = gpd['demand_z'].to_numpy()

    plt.plot(x, y, linewidth = 1.8, alpha = 0.35)

plt.title(f"Normalized demand trend (robust min-max)  |  parts shown = {len(parts)}")
plt.xlabel("yyyyww")
plt.ylabel("demand_mm_robust (0~1)")
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_norm.filter(pl.col('oper_part_no') == '0092200')